In [1]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from PIL import Image

In [2]:
# =============================================================================
# 1. CONFIGURATION
# =============================================================================

RESULTS_DIR = "."                                   # folder with the *_results.xlsx files
FIG_DIR     = "figures"                             # output folder for all figures
LAYOUT_DIR  = os.path.join(FIG_DIR, "layouts")
VARIANT     = "BASE"                                # {Problem Variants} field of the archetype code

os.makedirs(LAYOUT_DIR, exist_ok=True)

In [3]:
# =============================================================================
# 2. ARCHETYPE PILLARS
#    Archetype code: {Generation}_{Topology}_{Scale}_{Variant}
# =============================================================================
ARCHETYPES = {
    #  id          generation   topology       scale           generation method
    "3G_R_S": dict(gen="3GDH",  topo="Radial", scale="Small",  method="I"),
    "3G_L_S": dict(gen="3GDH",  topo="Ring",   scale="Small",  method="I"),
    "3G_M_S": dict(gen="3GDH",  topo="Meshed", scale="Small",  method="I"),
    "3G_M_M": dict(gen="3GDH",  topo="Meshed", scale="Medium", method="II"),
    "4G_M_M": dict(gen="4GDH",  topo="Meshed", scale="Medium", method="II"),
    "5G_M_M": dict(gen="5GDHC", topo="Meshed", scale="Medium", method="II"),
    "3G_M_L": dict(gen="3GDH",  topo="Meshed", scale="Large",  method="II"),
}

CLASS_NAMES = {
    "THV": "Thermo-Hydraulic Validation",
    "TA":  "Technology Assessment",
    "CS":  "Computational Scalability",
}

CS_SERIES = {
    "topology": ["3G_R_S", "3G_L_S", "3G_M_S"],     # small scale, topology varies
    "size":     ["3G_M_S", "3G_M_M", "3G_M_L"],     # meshed topology, scale varies
}
BENCH_OBJECTIVES = {
    "THV": ["3G_R_S", "3G_L_S", "3G_M_S"],          # topology effect
    "TA":  ["3G_M_M", "4G_M_M", "5G_M_M"],          # generation effect
    "CS":  list(dict.fromkeys(CS_SERIES["topology"] + CS_SERIES["size"])),
}

def folder_name(aid):
    """Archetype folder / file prefix, e.g. '3G_R_S' -> '3G_R_S_BASE'."""
    return f"{aid}_{VARIANT}"

# Overview table: archetype descriptors + class membership
overview = pd.DataFrame(ARCHETYPES).T
for cls, ids in BENCH_OBJECTIVES.items():
    overview[cls] = ["✓" if a in ids else "" for a in overview.index]
overview

,gen,topo,scale,method,THV,TA,CS
3G_R_S,3GDH,Radial,Small,I,✓,,✓
3G_L_S,3GDH,Ring,Small,I,✓,,✓
3G_M_S,3GDH,Meshed,Small,I,✓,,✓
3G_M_M,3GDH,Meshed,Medium,II,,✓,✓
4G_M_M,4GDH,Meshed,Medium,II,,✓,
5G_M_M,5GDHC,Meshed,Medium,II,,✓,
3G_M_L,3GDH,Meshed,Large,II,,,✓


In [4]:
# =============================================================================
# 3. LOAD SIMULATION RESULTS
#    One Excel workbook per archetype, one sheet per variable,
#    rows = timesteps (first column), columns = network elements.
# =============================================================================
LONG_FORMAT_SHEETS = {"convergence_traj"}            # sheets not indexed by timestep

def _realign_source(sheets, reference="sub_Q_W"):
    """Align the index of the 'source' sheet to the reference sheet, if needed."""
    if "source" not in sheets or reference not in sheets:
        return sheets
    src, ref_idx = sheets["source"], sheets[reference].index
    if len(src) != len(ref_idx):
        print(f"   [WARN] 'source' has {len(src)} rows vs {len(ref_idx)} in '{reference}': not realigned")
    elif not src.index.equals(ref_idx):
        sheets["source"] = src.set_axis(ref_idx).rename_axis("timestep")
        print(f"   [FIX] 'source' index realigned to '{reference}'")
    return sheets

def load_results(aid):
    """Read all sheets of an archetype workbook; index time-series sheets by timestep."""
    path = os.path.join(RESULTS_DIR, f"{folder_name(aid)}_results.xlsx")
    if not os.path.isfile(path):
        print(f"[SKIP] {aid}: {os.path.basename(path)} not found")
        return None
    sheets = pd.read_excel(path, sheet_name=None)
    for name, df in sheets.items():
        if name not in LONG_FORMAT_SHEETS:
            sheets[name] = df.rename(columns={df.columns[0]: "timestep"}).set_index("timestep")
    print(f"[OK]   {aid}: {len(sheets)} sheets")
    return _realign_source(sheets)

results = {}
for aid in ARCHETYPES:
    data = load_results(aid)
    if data is not None:
        results[aid] = data

[OK]   3G_R_S: 17 sheets
   [FIX] 'source' index realigned to 'sub_Q_W'
[OK]   3G_L_S: 17 sheets
   [FIX] 'source' index realigned to 'sub_Q_W'
[OK]   3G_M_S: 17 sheets
   [FIX] 'source' index realigned to 'sub_Q_W'
[OK]   3G_M_M: 15 sheets
   [FIX] 'source' index realigned to 'sub_Q_W'
[OK]   4G_M_M: 18 sheets
[OK]   5G_M_M: 31 sheets
[OK]   3G_M_L: 15 sheets
   [FIX] 'source' index realigned to 'sub_Q_W'


In [5]:
# =============================================================================
# 4. NETWORKS GROUPED BY BENCHMARK OBJECTIVE
# =============================================================================
def subset(ids):
    """Results of the given archetypes, in the given order (missing ones skipped)."""
    return {a: results[a] for a in ids if a in results}

THV_NETWORKS = subset(BENCH_OBJECTIVES["THV"])
TA_NETWORKS  = subset(BENCH_OBJECTIVES["TA"])
CS_NETWORKS  = subset(BENCH_OBJECTIVES["CS"])
CS_TOPOLOGY  = subset(CS_SERIES["topology"])
CS_SIZE      = subset(CS_SERIES["size"])

for cls, nets in (("THV", THV_NETWORKS), ("TA", TA_NETWORKS), ("CS", CS_NETWORKS)):
    print(f"{cls:<4} {CLASS_NAMES[cls]:<28} -> {list(nets)}")
print(f"     CS topology series           -> {list(CS_TOPOLOGY)}")
print(f"     CS size series               -> {list(CS_SIZE)}")

THV  Thermo-Hydraulic Validation  -> ['3G_R_S', '3G_L_S', '3G_M_S']
TA   Technology Assessment        -> ['3G_M_M', '4G_M_M', '5G_M_M']
CS   Computational Scalability    -> ['3G_R_S', '3G_L_S', '3G_M_S', '3G_M_M', '3G_M_L']
     CS topology series           -> ['3G_R_S', '3G_L_S', '3G_M_S']
     CS size series               -> ['3G_M_S', '3G_M_M', '3G_M_L']


In [12]:
# =============================================================================
# 5. NETWORK LAYOUTS — one PNG per archetype
# =============================================================================
# ---- Canvas and export ----------------------------------------------------
CANVAS_W_IN, CANVAS_H_IN = 3.0, 2.4      # drawing canvas [in]
DISPLAY_W_IN   = 0.74                    # width at which the image is shown in the paper [in]
MARGIN         = 0.05                    # relative margin around the network
EXPORT_DPI     = 600
TRANSPARENT    = True
SHOW_JUNCTIONS = True

# ---- Style (sizes refer to the displayed width) ---------------------------
LAYOUT_STYLE = {
    "pipe":       "black",
    "source":     dict(fc="blue",    ec="blue",    s=3.0,  z=6),
    "prosumer":   dict(fc="#069494", ec="#069494", s=1.0,  z=4),
    "substation": dict(fc="#CCCCFF", ec="blue",    s=1.0,  z=4),
    "junction":   dict(fc="black",   ec="black",   s=0.25, z=3),
}
PIPE_LW, EDGE_LW = 0.15, 0.10
FLIP_Y = {"I": True, "II": False}        # Methodology I coordinates have y pointing down

# ---- Archetype folder (nodes.csv / pipes.csv) -----------------------------
def locate_archetypes_dir(anchor=folder_name("3G_R_S"), max_up=5):
    cur = os.path.abspath(RESULTS_DIR)
    for _ in range(max_up + 1):
        hits = glob.glob(os.path.join(cur, "**", anchor, "nodes.csv"), recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
        cur = os.path.dirname(cur)
    raise FileNotFoundError("Set ARCHETYPES_DIR manually (folder containing the *_BASE folders).")

ARCHETYPES_DIR = globals().get("ARCHETYPES_DIR") or locate_archetypes_dir()
print("ARCHETYPES_DIR =", ARCHETYPES_DIR)

def load_topology(aid):
    """Node positions, node types and pipe table of an archetype."""
    base  = os.path.join(ARCHETYPES_DIR, folder_name(aid))
    nodes = pd.read_csv(os.path.join(base, "nodes.csv"))
    pipes = pd.read_csv(os.path.join(base, "pipes.csv"))
    sign  = -1.0 if FLIP_Y[ARCHETYPES[aid]["method"]] else 1.0
    pos   = {r.node_id: (float(r.x_m), sign * float(r.y_m)) for r in nodes.itertuples()}
    unknown = set(nodes.node_type) - set(LAYOUT_STYLE)
    if unknown:
        print(f"   [WARN] {aid}: node types {unknown} drawn as substations")
    ntype = {r.node_id: (r.node_type if r.node_type in LAYOUT_STYLE else "substation")
             for r in nodes.itertuples()}
    return pos, ntype, pipes

def fit_limits(xs, ys):
    """Axis limits centring the network on the canvas with equal x/y scale."""
    w, h = max(max(xs) - min(xs), 1e-9), max(max(ys) - min(ys), 1e-9)
    s  = min((1 - 2 * MARGIN) * CANVAS_W_IN / w, (1 - 2 * MARGIN) * CANVAS_H_IN / h)
    cx, cy = (max(xs) + min(xs)) / 2, (max(ys) + min(ys)) / 2
    return ((cx - CANVAS_W_IN / s / 2, cx + CANVAS_W_IN / s / 2),
            (cy - CANVAS_H_IN / s / 2, cy + CANVAS_H_IN / s / 2))

def draw_layout(aid):
    """Bare network layout on the fixed canvas."""
    pos, ntype, pipes = load_topology(aid)
    k = CANVAS_W_IN / DISPLAY_W_IN               # scale factor canvas -> displayed size

    fig = plt.figure(figsize=(CANVAS_W_IN, CANVAS_H_IN))
    ax  = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    xlim, ylim = fit_limits([p[0] for p in pos.values()], [p[1] for p in pos.values()])
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)

    ax.add_collection(LineCollection([(pos[r.from_node], pos[r.to_node]) for r in pipes.itertuples()],
                                     colors=LAYOUT_STYLE["pipe"], linewidths=PIPE_LW * k,
                                     capstyle="round", zorder=2))
    for kind in ("junction", "substation", "prosumer", "source"):
        if kind == "junction" and not SHOW_JUNCTIONS:
            continue
        ids = [n for n, t in ntype.items() if t == kind]
        if ids:
            st = LAYOUT_STYLE[kind]
            ax.scatter([pos[n][0] for n in ids], [pos[n][1] for n in ids], s=st["s"] * k**2,
                       facecolor=st["fc"], edgecolor=st["ec"], linewidths=EDGE_LW * k, zorder=st["z"])
    return fig

# ---- Export ---------------------------------------------------------------
LAYOUT_FILES = {}
for aid in ARCHETYPES:
    if not os.path.isdir(os.path.join(ARCHETYPES_DIR, folder_name(aid))):
        print(f"[SKIP] {aid}: folder not found")
        continue
    fig  = draw_layout(aid)
    path = os.path.join(LAYOUT_DIR, f"{aid}_layout.png")
    fig.savefig(path, dpi=EXPORT_DPI, transparent=TRANSPARENT)   # no bbox_inches: same canvas for all
    plt.close(fig)
    LAYOUT_FILES[aid] = path
    print(f"[SAVED] {aid} -> {path}")

# ---- Consistency check: identical pixel size ------------------------------
sizes = {a: Image.open(p).size for a, p in LAYOUT_FILES.items()}
assert len(set(sizes.values())) <= 1, f"Layouts do not share the same pixel size: {sizes}"
print("All layouts share the same size:", next(iter(sizes.values()), "n/a"))

ARCHETYPES_DIR = C:\Users\sara.ferrero\PycharmProjects\dhn-benchmark\archetypes_generation\archetypes
[SAVED] 3G_R_S -> figures\layouts\3G_R_S_layout.png
[SAVED] 3G_L_S -> figures\layouts\3G_L_S_layout.png
[SAVED] 3G_M_S -> figures\layouts\3G_M_S_layout.png
[SAVED] 3G_M_M -> figures\layouts\3G_M_M_layout.png
[SAVED] 4G_M_M -> figures\layouts\4G_M_M_layout.png
[SAVED] 5G_M_M -> figures\layouts\5G_M_M_layout.png
[SAVED] 3G_M_L -> figures\layouts\3G_M_L_layout.png
All layouts share the same size: (1800, 1440)
